In [ ]:
import pandas as pd
from modelens import RegressionAnalyzer

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [ ]:
raw_data = pd.read_csv('dataset.csv')
df = raw_data.copy()

In [ ]:
# raw_data.info()

In [ ]:
df = df.rename(columns={
    "X1": "Relative Compactness",
    "X2": "Surface Area",
    "X3": "Wall Area",
    "X4": "Roof Area",
    "X5": "Overall Height",
    "X6": "Orientation",
    "X7": "Glazing Area",
    "X8": "Glazing Area Distribution",
    "Y1": "Heating Load",
    "Y2": "Cooling Load",
})

In [ ]:
analyzer = RegressionAnalyzer(df=df,target='Heating Load')

In [ ]:
# analyzer.info()

In [ ]:
target = 'Heating Load'
X = df.drop(columns=[target,'Cooling Load'])
y = df[target]
features = df.columns.drop([target,'Cooling Load'])

#### Comaring Models

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import (
    AdaBoostRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    Ridge,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures


models = {
    # Linear
    "Linear Regression": LinearRegression(),
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge()),
    ]),

    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso()),
    ]),

    "ElasticNet": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet()),
    ]),

    "Polynomial Regression D2": Pipeline([
    ("polynomial",PolynomialFeatures(degree=2,include_bias=False)),
     ("scaler", StandardScaler()),
    ("model",LinearRegression()),
    ]),

    # Distance / Kernel
    "KNN": Pipeline([
        ("scaler",StandardScaler()),
        ("KNN",KNeighborsRegressor())
    ]),
    
    "SVR":Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR()),
    ]),

    # Tree
    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
    ),

    # Bagging
    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees": ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    # Boosting
    "AdaBoost": AdaBoostRegressor(
        random_state=42,
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        random_state=42,
    ),

    "XGBoost": XGBRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    "LightGBM": LGBMRegressor(
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    ),

    "CatBoost": CatBoostRegressor(
        random_state=42,
        verbose=0,
    ),
}
# compare = analyzer.compare_models(models=models,features=features,export_html=True)

#### Model Selection for Tuning

***1. Ridge***

***2. Polynomial Regression***

***3. Decision Tree***

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge()),
])


param_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        0.5,
        1.0,
        2.0,
        5.0,
        10.0,
        50.0,
        100.0,
    ]
}


# analyzer.tune_model(
#     model=model,
#     features=features,
#     param_grid=param_grid,
#     export_html=True,
#     html_path='ridge'
# )

In [ ]:
model = Pipeline([
    ("polynomial",PolynomialFeatures(include_bias=False)),
    ("scaler",StandardScaler()),
    ("model",LinearRegression()),
])


param_grid = {
    "polynomial__degree": [
        2,
        3,
        4,
    ]
}


# analyzer.tune_model(
#     model=model,
#     features=features,
#     param_grid=param_grid,
#     export_html=True,
#     html_path='polynomial'
# )

In [ ]:
model = DecisionTreeRegressor(random_state=42)


param_grid = {
    "max_depth": [
        None,
        3,
        5,
        10,
        20,
    ],
    "min_samples_split": [
        2,
        5,
        10,
    ],
    "min_samples_leaf": [
        1,
        2,
        4,
    ],
}


# analyzer.tune_model(
#     model=model,
#     features=features,
#     param_grid=param_grid,
#     export_html=True,
#     html_path="decision_tree",
# )

In [ ]:
model = CatBoostRegressor(
    random_state=42,
    verbose=0,
)


param_grid = {
    "iterations": [
        300,
        500,
        1000,
    ],
    "depth": [
        4,
        6,
        8,
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.1,
    ],
    "l2_leaf_reg": [
        1,
        3,
        5,
    ],
}


# analyzer.tune_model(
#     model=model,
#     features=features,
#     param_grid=param_grid,
#     export_html=True,
#     html_path="catboost",
# )

#### CatBoostRegreesor Class is Selected

In [ ]:
_,suspicious_features=analyzer.correlation(features=features)

#### Decrease Feature

In [ ]:
# analyzer.vif(features=features)

In [ ]:
selected_model = CatBoostRegressor(depth= 8, iterations= 1000, l2_leaf_reg= 1, learning_rate= 0.1)
analyzer.evaluate_single_feature_removal(model=selected_model,features=features,export_html=True)

In [ ]:
# analyzer.evaluate_feature_removal_combinations(model=selected_model,features=features,candidates=suspicious_features,export_html=True)

In [ ]:
selected_features = ['Surface Area', 'Wall Area', 'Orientation', 'Glazing Area', 'Glazing Area Distribution']

analyzer.compare_models(
    models={
        "CatBoost All Features": selected_model,
    },
    features=features,
    export_html=True,
    html_path="catboost_all_features",
)

analyzer.compare_models(
    models={
        "CatBoost 5 Features": selected_model,
    },
    features=selected_features,
    export_html=True,
    html_path="catboost_5_features",
)

In [ ]:
analyzer.residual_analysis(model=selected_model,export_html=True)

In [ ]:
analyzer.learning_curve(model=selected_model)